In [ ]:
!pip install catboost -q


## 1. Install & Import Dependencies

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

warnings.filterwarnings('ignore')
np.random.seed(42)

# ML — Preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# ML — Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

# ML Evaluation
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, f1_score, ConfusionMatrixDisplay
)

# Imbalance handling
from imblearn.over_sampling import SMOTE

# Explainability
import shap

print('All libraries imported successfully')


## 2. Data Loading

In [ ]:

DATA_PATH = 'data/Students Social Media Addiction.csv'

df = pd.read_csv(DATA_PATH)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:

print('=== Dataset Info ===')
df.info()
print('\n=== Statistical Summary ===')
df.describe().round(2)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
#3.1 Missing Values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

print('=== Missing Values ===')
print(missing_df[missing_df['Missing Count'] > 0] if missing_df['Missing Count'].sum() > 0 else 'No missing values')

In [ ]:
#3.2 Target Variable Distribution (Addicted Score (4 classes))
def bin_addiction(score):
    if score <= 3:   return 0  # Low
    elif score <= 5: return 1  # Moderate
    elif score <= 7: return 2  # High
    else:            return 3  # Severe

df['Addiction_Level'] = df['Addicted_Score'].apply(bin_addiction)
label_map = {0: 'Low', 1: 'Moderate', 2: 'High', 3: 'Severe'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Target Variable Analysis', fontsize=14, fontweight='bold')

#Addicted_Score distribution
axes[0].hist(df['Addicted_Score'], bins=20, color='#6366F1', edgecolor='white', alpha=0.85)
axes[0].set_title('Addicted Score Distribution')
axes[0].set_xlabel('Addicted Score (2-9)')
axes[0].set_ylabel('Frequency')


ordered_labels = ['Low', 'Moderate', 'High', 'Severe']
ordered_colors = ['#22C55E', '#F59E0B', '#F97316', '#EF4444']
ordered_counts = [
    (df['Addiction_Level'].map(label_map) == label).sum()
    for label in ordered_labels
]

axes[1].bar(ordered_labels, ordered_counts, color=ordered_colors, edgecolor='white')
axes[1].set_title('Addiction Level Class Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
for i, (label, val) in enumerate(zip(ordered_labels, ordered_counts)):
    axes[1].text(i, val + 3, f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClass distribution:')
for k, v in label_map.items():
    cnt = (df['Addiction_Level'] == k).sum()
    print(f'  {v:10s} (class {k}): {cnt:4d}  ({cnt/len(df)*100:.1f}%)')

In [ ]:

df['Affects_Academic_Performance'] = df['Affects_Academic_Performance'].map({'Yes': 1, 'No': 0})

In [ ]:
#3.3 Correlation Heatmap
numeric_cols = ['Age', 'Avg_Daily_Usage_Hours', 'Affects_Academic_Performance',
                'Sleep_Hours_Per_Night', 'Mental_Health_Score',
                'Conflicts_Over_Social_Media', 'Addicted_Score']

plt.figure(figsize=(10, 7))
corr = df[numeric_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey correlations with Addicted_Score:')
print(corr['Addicted_Score'].drop('Addicted_Score').sort_values(key=abs, ascending=False).round(3))

In [ ]:
#3.4 Feature Distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Feature Distributions by Addiction Level', fontsize=14, fontweight='bold')

level_order  = [0, 1, 2, 3]
level_labels = ['Low', 'Moderate', 'High', 'Severe']
palette      = ['#22C55E', '#F59E0B', '#F97316', '#EF4444']

features = ['Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night', 'Mental_Health_Score',
            'Conflicts_Over_Social_Media', 'Age', 'Affects_Academic_Performance']

for ax, feat in zip(axes.flat, features):
    for level, color, lbl in zip(level_order, palette, level_labels):
        data = df[df['Addiction_Level'] == level][feat]
        ax.hist(data, bins=15, alpha=0.55, color=color, label=lbl, edgecolor='none')
    ax.set_title(feat.replace('_', ' '))
    ax.set_xlabel('')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#3.5 Categorical EDA
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Categorical Feature Analysis', fontsize=13, fontweight='bold')

level_labels = ['Low', 'Moderate', 'High', 'Severe']
palette      = ['#22C55E', '#F59E0B', '#F97316', '#EF4444']

cat_cols = ['Gender', 'Most_Used_Platform', 'Academic_Level']
for ax, col in zip(axes, cat_cols):
    ct     = pd.crosstab(df[col], df['Addiction_Level'].map(label_map))
    ct     = ct.reindex(columns=level_labels, fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind='bar', ax=ax, color=palette, edgecolor='white', width=0.75)
    ax.set_title(col.replace('_', ' '))
    ax.set_xlabel('')
    ax.set_ylabel('% within group')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.legend(title='Addiction Level', fontsize=7)

plt.tight_layout()
plt.savefig('categorical_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
#4.1 Encode Categoricals
df_processed = df.copy()

cat_features = ['Gender', 'Academic_Level', 'Country', 'Most_Used_Platform', 'Relationship_Status']
label_encoders = {}

for col in cat_features:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le
    print(f'  {col:30s} → {dict(zip(le.classes_, le.transform(le.classes_)))}')

joblib.dump(label_encoders, 'models/label_encoders.pkl')

In [ ]:
#4.2 Feature / Target Split
FEATURE_COLS = [
    'Age', 'Gender', 'Academic_Level', 'Country',
    'Avg_Daily_Usage_Hours', 'Most_Used_Platform',
    'Affects_Academic_Performance', 'Sleep_Hours_Per_Night',
    'Mental_Health_Score', 'Relationship_Status',
    'Conflicts_Over_Social_Media'
]

X = df_processed[FEATURE_COLS]
y = df_processed['Addiction_Level']

print(f'X shape: {X.shape}')
print(f'y distribution:\n{y.value_counts().sort_index()}')

In [ ]:
#4.3  Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
#4.4 Handle Class Imbalance with SMOTE
print('Before SMOTE:', dict(y_train.value_counts().sort_index()))

smote = SMOTE(random_state=42, k_neighbors=3)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('After  SMOTE:', dict(pd.Series(y_train_bal).value_counts().sort_index()))
print(f'\nTrain size after SMOTE: {X_train_bal.shape}')

In [ ]:
# 4.5 Feature Scaling (for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled  = scaler.transform(X_test)

joblib.dump(scaler, 'models/scaler.pkl')
print('Scaler fitted and saved')

## 5. Model Training

In [ ]:
# 5.1 Models
models = {
    'XGBoost': XGBClassifier( n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    ),
    'CatBoost': CatBoostClassifier( iterations=300, depth=6, learning_rate=0.05, loss_function='MultiClass',
        random_seed=42,
        verbose=0
    ),
    'LightGBM': LGBMClassifier( n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    'Random Forest': RandomForestClassifier( n_estimators=200, max_depth=10, min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ),
    'Logistic Regression': LogisticRegression( max_iter=1000, multi_class='multinomial', solver='lbfgs',
        random_state=42,
        n_jobs=-1
    )
}

print(f'{len(models)} models defined')

In [ ]:
#5.2 Train All Models + Cross-Validation
from sklearn.model_selection import StratifiedKFold

results = {}
trained_models = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f'\nTraining {name}...')

    # Use scaled data only for Logistic Regression
    X_tr = X_train_scaled if name == 'Logistic Regression' else X_train_bal
    X_te = X_test_scaled  if name == 'Logistic Regression' else X_test

    model.fit(X_tr, y_train_bal)
    trained_models[name] = model

    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)


    cv_scores = cross_val_score(model, X_tr, y_train_bal, cv=skf, scoring='accuracy', n_jobs=-1)

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    roc  = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')

    results[name] = {
        'Accuracy': round(acc, 4),
        'F1 Score': round(f1, 4),
        'ROC-AUC':  round(roc, 4),
        'CV Mean':  round(cv_scores.mean(), 4),
        'CV Std':   round(cv_scores.std(), 4),
        'y_pred':   y_pred,
        'y_prob':   y_prob
    }

    print(f'   Accuracy: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {roc:.4f} | CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

print('\nAll models trained successfully')

## 6. Model Evaluation

In [ ]:
#6.1 Summary Table
summary = pd.DataFrame({
    name: {
        'Accuracy': res['Accuracy'],
        'F1 Score': res['F1 Score'],
        'ROC-AUC':  res['ROC-AUC'],
        'CV Mean':  res['CV Mean'],
        'CV Std':   res['CV Std']
    }
    for name, res in results.items()
}).T.sort_values('Accuracy', ascending=False)

print('=== MODEL PERFORMANCE SUMMARY ===')
print(summary.to_string())

best_model_name = summary['Accuracy'].idxmax()
print(f'\nBest model: {best_model_name} (Accuracy: {summary.loc[best_model_name, "Accuracy"]})')
##%%


In [ ]:
#6.2 Performance Bar Charts
metrics = ['Accuracy', 'F1 Score', 'ROC-AUC']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison - Key Metrics', fontsize=13, fontweight='bold')

bar_colors = ['#6366F1', '#06B6D4', '#10B981', '#F59E0B', '#EF4444']
model_names = list(summary.index)

for ax, metric in zip(axes, metrics):
    vals = [results[m][metric] for m in model_names]
    bars = ax.barh(model_names, vals, color=bar_colors, edgecolor='white', height=0.6)
    ax.set_xlim(0, 1.05)
    ax.set_title(metric)
    ax.set_xlabel('Score')
    for bar, val in zip(bars, vals):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
                va='center', fontsize=9)

plt.tight_layout()
plt.savefig('05_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#6.3 Confusion Matrices
class_names = ['Low', 'Moderate', 'High', 'Severe']
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('Confusion Matrices - All Models', fontsize=13, fontweight='bold')

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('06_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#6.4 Classification Report (Best Model)
print(f'=== Classification Report: {best_model_name} ===')
print(classification_report(
    y_test,
    results[best_model_name]['y_pred'],
    target_names=class_names
))

In [ ]:
#6.5 Feature Importance (Random Forest)
random_forest_model = trained_models['Random Forest']
importances = pd.Series(
    random_forest_model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(importances.index, importances.values, color='#6366F1', edgecolor='white')
plt.title('Random Forest Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
for bar, val in zip(bars, importances.values):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('07_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#6.6 SHAP Explainability Random Forest (Best Model)
print('Computing SHAP values for Random Forest (this may take ~30s)...')
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

explainer   = shap.TreeExplainer(random_forest_model)   # ← correct model
shap_values = explainer.shap_values(X_test)

fig = plt.figure(figsize=(18, 7))

shap.summary_plot(
    shap_values,
    X_test,
    feature_names = FEATURE_COLS,
    class_names   = class_names,
    show          = False,
    plot_size     = None,
    color_bar     = True,
)
fig.suptitle(
    'SHAP Feature Impact — Random Forest',   # ← fixed title
    fontsize   = 14,
    fontweight = 'bold',
    y          = 1.03,
)

plt.subplots_adjust(top=0.88, bottom=0.12, wspace=0.35)

plt.savefig(
    '08_shap_summary.png',
    dpi         = 150,
    bbox_inches = 'tight',
    pad_inches  = 0.4,
)
plt.show()
print('SHAP analysis complete')

In [ ]:
#6.7 Hyperparameter Tuning (XGBoost, Random Forest, LightGBM)


from sklearn.model_selection import RandomizedSearchCV
import warnings
warnings.filterwarnings('ignore')

print('Hyperparameter tuning...')


xgb_param_grid = {
    'n_estimators':     [200, 300, 400],
    'max_depth':        [4, 5, 6, 7],
    'learning_rate':    [0.03, 0.05, 0.1],
    'subsample':        [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma':            [0, 0.1, 0.2],
}


rf_param_grid = {
    'n_estimators':     [100, 200, 300, 400],
    'max_depth':        [6, 8, 10, 12, None],
    'min_samples_split':[2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features':     ['sqrt', 'log2', 0.5],
    'bootstrap':        [True, False],
}


lgbm_param_grid = {
    'n_estimators':     [200, 300, 400],
    'max_depth':        [4, 5, 6, 7],
    'learning_rate':    [0.03, 0.05, 0.1],
    'num_leaves':       [15, 31, 50, 63],
    'subsample':        [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_samples':[10, 20, 30],
    'reg_alpha':        [0, 0.1, 0.5],
    'reg_lambda':       [0, 0.1, 1.0],
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Run RandomizedSearchCV for each model
tuning_configs = {
    'XGBoost': {
        'model': XGBClassifier(
            use_label_encoder=False,
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=-1),

        'params': xgb_param_grid,
        'X': X_train_bal,
        'X_test': X_test,
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': rf_param_grid,
        'X': X_train_bal,
        'X_test': X_test,
    },
    'LightGBM': {
        'model': LGBMClassifier(
            objective='multiclass',
            num_class=4,
            random_state=42,
            n_jobs=-1,
            verbose=-1),

        'params': lgbm_param_grid,
        'X': X_train_bal,
        'X_test': X_test,
    },
}

tuned_models  = {}
tuned_results = {}

for name, cfg in tuning_configs.items():
    print(f'Tuning {name}...')

    search = RandomizedSearchCV(
        estimator          = cfg['model'],
        param_distributions= cfg['params'],
        n_iter             = 30,
        cv                 = skf,
        scoring            = 'accuracy',
        n_jobs             = -1,
        random_state       = 42,
        verbose            = 0,
    )

    search.fit(cfg['X'], y_train_bal)

    best = search.best_estimator_
    tuned_models[name] = best

    y_pred = best.predict(cfg['X_test'])
    y_prob = best.predict_proba(cfg['X_test'])

    cv_scores = cross_val_score(
        best, cfg['X'], y_train_bal,
        cv=skf, scoring='accuracy', n_jobs=-1
    )

    tuned_results[name] = {
        'Accuracy':    round(accuracy_score(y_test, y_pred), 4),
        'F1 Score':    round(f1_score(y_test, y_pred, average='weighted'), 4),
        'ROC-AUC':     round(roc_auc_score(y_test, y_prob,
                             multi_class='ovr', average='weighted'), 4),
        'CV Mean':     round(cv_scores.mean(), 4),
        'CV Std':      round(cv_scores.std(), 4),
        'Best Params': search.best_params_,
        'y_pred':      y_pred,
        'y_prob':      y_prob,
    }

    print(f'  Best params : {search.best_params_}')
    print(f'  Accuracy    : {tuned_results[name]["Accuracy"]:.4f}')
    print(f'  F1 Score    : {tuned_results[name]["F1 Score"]:.4f}')
    print(f'  ROC-AUC     : {tuned_results[name]["ROC-AUC"]:.4f}')
    print(f'  CV Mean     : {tuned_results[name]["CV Mean"]:.4f} '
          f'± {tuned_results[name]["CV Std"]:.4f}\n')

print('Hyperparameter tuning complete.')

In [ ]:
#6.8 Tuned vs Non-Tuned Comparison Table


tune_models_list = ['XGBoost', 'Random Forest', 'LightGBM']
metrics_list     = ['Accuracy', 'F1 Score', 'ROC-AUC', 'CV Mean']

rows = []
for name in tune_models_list:
    # Before tuning from original results dict
    before = results[name]
    rows.append({
        'Model':    name,
        'Version':  'Before Tuning',
        'Accuracy': before['Accuracy'],
        'F1 Score': before['F1 Score'],
        'ROC-AUC':  before['ROC-AUC'],
        'CV Mean':  before['CV Mean'],
    })
    # After tuning
    after = tuned_results[name]
    rows.append({
        'Model':    name,
        'Version':  'After Tuning',
        'Accuracy': after['Accuracy'],
        'F1 Score': after['F1 Score'],
        'ROC-AUC':  after['ROC-AUC'],
        'CV Mean':  after['CV Mean'],
    })

compare_df = pd.DataFrame(rows)
print('=== TUNED vs NON-TUNED COMPARISON ===\n')
print(compare_df.to_string(index=False))


fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Hyperparameter Tuning - Before vs After Comparison',
             fontsize=14, fontweight='bold')

colors_before = ['#94A3B8', '#94A3B8', '#94A3B8']
colors_after  = ['#6366F1', '#10B981', '#F59E0B']

x     = np.arange(len(tune_models_list))
width = 0.35

for ax, metric in zip(axes, metrics_list):
    before_vals = [results[m][metric]      for m in tune_models_list]
    after_vals  = [tuned_results[m][metric] for m in tune_models_list]

    bars1 = ax.bar(x - width/2, before_vals, width,
                   label='Before Tuning', color='#94A3B8',
                   edgecolor='white', alpha=0.9)
    bars2 = ax.bar(x + width/2, after_vals,  width,
                   label='After Tuning',
                   color=['#6366F1', '#10B981', '#F59E0B'],
                   edgecolor='white', alpha=0.95)

    ax.set_title(metric, fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['XGBoost', 'RF', 'LGBM'], fontsize=9)
    ax.set_ylim(0.90, 1.02)
    ax.set_ylabel('Score')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)


    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                f'{h:.4f}', ha='center', va='bottom', fontsize=7)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                f'{h:.4f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('09_tuning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 6.9 ROC Curves Tuned Models (One vs Rest per class)


from sklearn.preprocessing import label_binarize

class_names  = ['Low', 'Moderate', 'High', 'Severe']
n_classes    = 4
y_test_bin   = label_binarize(y_test, classes=[0, 1, 2, 3])

model_colors = {
    'XGBoost':       '#6366F1',
    'Random Forest': '#10B981',
    'LightGBM':      '#F59E0B',
}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('ROC Curves - Tuned Models (One-vs-Rest per class)',
             fontsize=14, fontweight='bold')

from sklearn.metrics import roc_curve, auc

for ax, (cls_idx, cls_name) in enumerate(zip(range(n_classes), class_names)):
    for model_name, color in model_colors.items():
        y_prob_col = tuned_results[model_name]['y_prob'][:, cls_idx]
        fpr, tpr, _ = roc_curve(y_test_bin[:, cls_idx], y_prob_col)
        roc_auc_val  = auc(fpr, tpr)
        axes[ax].plot(fpr, tpr, color=color, lw=2,
                      label=f'{model_name} (AUC={roc_auc_val:.4f})')

    axes[ax].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    axes[ax].fill_between([0, 1], [0, 1], alpha=0.04, color='grey')
    axes[ax].set_title(f'Class: {cls_name}', fontsize=11, fontweight='bold')
    axes[ax].set_xlabel('False Positive Rate')
    axes[ax].set_ylabel('True Positive Rate')
    axes[ax].legend(fontsize=8, loc='lower right')
    axes[ax].grid(alpha=0.25)
    axes[ax].set_xlim([0.0, 1.0])
    axes[ax].set_ylim([0.0, 1.02])

plt.tight_layout()
plt.savefig('10_roc_curves_tuned.png', dpi=150, bbox_inches='tight')
plt.show()

## 6.10 Save All Model and Metadata

In [ ]:
#6.10  Save ALL Models + Metadata

import json, os

os.makedirs('models', exist_ok=True)


for name, model in trained_models.items():
    safe_name = name.lower().replace(' ', '_')
    path = f'models/{safe_name}_model.pkl'
    joblib.dump(model, path)
    print(f'  Saved → {path}')


if 'tuned_models' in dir():
    for name, model in tuned_models.items():
        safe = name.lower().replace(' ', '_')
        path = f'models/{safe}_tuned.pkl'
        joblib.dump(model, path)
        print(f'  Saved → {path}')


model_scores = {
    name: {
        'accuracy': res['Accuracy'],
        'f1':       res['F1 Score'],
        'roc_auc':  res['ROC-AUC'],
    }
    for name, res in results.items()
}

if 'tuned_results' in dir():
    for name, res in tuned_results.items():
        model_scores[f'{name} (Tuned)'] = {
            'accuracy': res['Accuracy'],
            'f1':       res['F1 Score'],
            'roc_auc':  res['ROC-AUC'],
        }

_all = {}

for name, res in results.items():
    _all[f'{name} (Non-Tuned)'] = {
        'model':      trained_models[name],
        'Accuracy':   res['Accuracy'],
        'F1 Score':   res['F1 Score'],
        'ROC-AUC':    res['ROC-AUC'],
        'CV Mean':    res['CV Mean'],
        'CV Std':     res['CV Std'],
        'y_pred':     res['y_pred'],
        'source':     'Non-Tuned',
        'base_name':  name,
    }

if 'tuned_results' in dir():
    for name, res in tuned_results.items():
        _all[f'{name} (Tuned)'] = {
            'model':      tuned_models[name],
            'Accuracy':   res['Accuracy'],
            'F1 Score':   res['F1 Score'],
            'ROC-AUC':    res['ROC-AUC'],
            'CV Mean':    res['CV Mean'],
            'CV Std':     res['CV Std'],
            'y_pred':     res['y_pred'],
            'source':     'Tuned',
            'base_name':  name,
        }


for key, info in _all.items():
    info['combined_score'] = (info['Accuracy'] * 0.60) + (info['F1 Score'] * 0.40)


best_key   = max(_all, key=lambda k: _all[k]['combined_score'])
best_info  = _all[best_key]
best_model = best_info['model']

print(f'\n  Best model identified: {best_key}')
print(f'  Accuracy : {best_info["Accuracy"]:.4f}')
print(f'  F1 Score : {best_info["F1 Score"]:.4f}')
print(f'  ROC-AUC  : {best_info["ROC-AUC"]:.4f}')


joblib.dump(best_model, 'models/addiction_model.pkl')
print(f'  Saved → models/addiction_model.pkl')


metadata = {
    'feature_cols':       FEATURE_COLS,
    'cat_features':       cat_features,
    'label_map':          {str(k): v for k, v in label_map.items()},
    'best_model':         best_key,
    'best_model_source':  best_info['source'],
    'best_model_scores':  {
        'accuracy': best_info['Accuracy'],
        'f1':       best_info['F1 Score'],
        'roc_auc':  best_info['ROC-AUC'],
        'cv_mean':  best_info['CV Mean'],
    },
    'model_scores':       model_scores,
}

with open('models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('\nAll models and metadata.json saved to models/')
print('Variables best_key, best_info, best_model are now available for 6.11 and 6.12.')

In [ ]:
#6.11 Full Summary Table All Models Before & After Tuning


print('=== COMPLETE RESULTS: ALL MODELS BEFORE & AFTER TUNING ===\n')

all_rows = []


for name, res in results.items():
    all_rows.append({
        'Model':    name,
        'Status':   'Non-Tuned',
        'Accuracy': res['Accuracy'],
        'F1 Score': res['F1 Score'],
        'ROC-AUC':  res['ROC-AUC'],
        'CV Mean':  res['CV Mean'],
        'CV Std':   res['CV Std'],
    })


for name, res in tuned_results.items():
    all_rows.append({
        'Model':    f'{name} (Tuned)',
        'Status':   'Tuned',
        'Accuracy': res['Accuracy'],
        'F1 Score': res['F1 Score'],
        'ROC-AUC':  res['ROC-AUC'],
        'CV Mean':  res['CV Mean'],
        'CV Std':   res['CV Std'],
    })

final_df = (
    pd.DataFrame(all_rows)
      .sort_values('Accuracy', ascending=False)
      .reset_index(drop=True)
)


final_df.index = final_df.index + 1
final_df.index.name = 'Rank'

print(final_df.to_string())


print(f'\nBest overall model : {best_key}')
print(f'Source             : {best_info["source"]}')
print(f'Accuracy           : {best_info["Accuracy"]:.4f}')
print(f'F1 Score           : {best_info["F1 Score"]:.4f}')
print(f'ROC-AUC            : {best_info["ROC-AUC"]:.4f}')
print(f'Production model   → models/addiction_model.pkl')

In [ ]:
#7. Sample Prediction


sample_input = {
    'Age':                          20,
    'Gender':                       'Female',
    'Academic_Level':               'Undergraduate',
    'Country':                      'Sri Lanka',
    'Avg_Daily_Usage_Hours':        7.5,
    'Most_Used_Platform':           'Instagram',
    'Affects_Academic_Performance': 1,
    'Sleep_Hours_Per_Night':        5.0,
    'Mental_Health_Score':          4.0,
    'Relationship_Status':          'Single',
    'Conflicts_Over_Social_Media':  3,
}

# Encode categoricals using saved label encoders
input_df = pd.DataFrame([sample_input])
for col in cat_features:
    le = label_encoders[col]
    input_df[col] = le.transform([sample_input[col]])

input_arr = input_df[FEATURE_COLS].values.astype(float)


if best_info['base_name'] == 'Logistic Regression':
    input_arr_use = scaler.transform(input_arr)
else:
    input_arr_use = input_arr


pred_class = int(best_model.predict(input_arr_use)[0])
pred_proba = best_model.predict_proba(input_arr_use)[0]

print('=== Sample Prediction ===')
print(f'Model used  : {best_key}')
print(f'Input       : {sample_input}')
print(f'\nPredicted Class : {label_map[pred_class]} (class {pred_class})')
print(f'Confidence      : {max(pred_proba)*100:.1f}%')
print(f'\nClass Probabilities:')
for cls, prob in zip(class_names, pred_proba):
    bar    = '█' * int(prob * 30)
    spaces = ' ' * (30 - int(prob * 30))
    print(f'  {cls:10s}: {bar}{spaces} {prob*100:.1f}%')

In [ ]:
#Resave XGBoost using native format (cross-version safe)

import os, json
os.makedirs('models', exist_ok=True)

if 'tuned_models' in dir() and 'XGBoost' in tuned_models:
    tuned_models['XGBoost'].save_model('models/xgboost_tuned.json')
    print('Saved → models/xgboost_tuned.json  (native XGBoost format)')


if 'XGBoost' in trained_models:
    trained_models['XGBoost'].save_model('models/xgboost_model.json')
    print('Saved → models/xgboost_model.json')


import joblib

joblib.dump(scaler,        'models/scaler.pkl')
joblib.dump(label_encoders,'models/label_encoders.pkl')
print('Resaved → scaler.pkl, label_encoders.pkl')


for name, model in trained_models.items():
    if name != 'XGBoost':
        safe = name.lower().replace(' ', '_')
        joblib.dump(model, f'models/{safe}_model.pkl')
        print(f'Resaved → models/{safe}_model.pkl')

if 'tuned_models' in dir():
    for name, model in tuned_models.items():
        if name != 'XGBoost':
            safe = name.lower().replace(' ', '_')
            joblib.dump(model, f'models/{safe}_tuned.pkl')
            print(f'Resaved → models/{safe}_tuned.pkl')

